# Heart Disease — Train Pipeline

Notebook này chạy toàn bộ pipeline ML:
1. **Preprocess** — làm sạch dữ liệu
2. **Normalize** — CSV đã scale sẵn, giữ nguyên
3. **Train** — train 8 classifier trên `raw_train.csv`
4. **Evaluate** — đánh giá trên `raw_val.csv`
5. **Save** — lưu model vào `models/`

Sau khi chạy xong → `streamlit run app.py` để dùng UI predict.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    VotingClassifier,
)
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
    print("⚠️  XGBoost không khả dụng — bỏ qua model này.")

⚠️  XGBoost không khả dụng — bỏ qua model này.


In [2]:
# ── Config ──────────────────────────────────────────────────────────────
BASE_DIR  = Path(".").resolve()
DATA_DIR  = BASE_DIR / "Data_raw"
MODEL_DIR = BASE_DIR / "models"

FEATURES = [
    "age", "trestbps", "chol", "thalach", "oldpeak",
    "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal",
]
TARGET = "target"

# Mean/std UCI — dùng để transform input bệnh nhân từ UI (giá trị thật)
CONT_STATS = {
    "age":      (54.37,  9.08),
    "trestbps": (131.62, 17.54),
    "chol":     (246.26, 51.83),
    "thalach":  (149.65, 22.91),
    "oldpeak":  (1.04,   1.16),
}

MODEL_NAMES = {
    "DecisionTreeClassifier":     "Decision Tree",
    "KNeighborsClassifier":       "K-NN",
    "GaussianNB":                 "Naive Bayes",
    "RandomForestClassifier":     "Random Forest",
    "AdaBoostClassifier":         "AdaBoost",
    "GradientBoostingClassifier": "Gradient Boosting",
    "XGBClassifier":              "XGBoost",
    "VotingClassifier":           "Ensemble (Soft Voting)",
}

## 1. Load & Preprocess

In [3]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    """Làm sạch: chọn cột, ép kiểu số, xoá duplicate, fill missing."""
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    df = df[FEATURES + [TARGET]].apply(pd.to_numeric, errors="coerce").drop_duplicates()
    df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())
    df[TARGET]   = df[TARGET].fillna(df[TARGET].mode()[0]).astype(int)
    return df


train_df = preprocess(pd.read_csv(DATA_DIR / "raw_train.csv"))
val_df   = preprocess(pd.read_csv(DATA_DIR / "raw_val.csv"))

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val,   y_val   = val_df[FEATURES],   val_df[TARGET]

print(f"Train: {X_train.shape[0]} samples  |  Val: {X_val.shape[0]} samples")
print(f"Target distribution (train):\n{y_train.value_counts()}")
train_df.head(3)

Train: 242 samples  |  Val: 30 samples
Target distribution (train):
target
0    131
1    111
Name: count, dtype: int64


,age,trestbps,chol,thalach,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,target
0,-0.729485,-0.395692,0.458139,0.708371,-0.445445,1.0,1.000000,0.0,1.0,0.0,0.5,0.000000,1.0,1
1,0.050166,-0.054513,0.230598,0.222495,-0.891627,1.0,0.333333,0.0,0.0,0.0,0.0,0.000000,0.0,0
2,-0.061212,0.059213,0.723605,0.399178,-0.891627,0.0,0.333333,1.0,1.0,1.0,0.0,0.333333,0.0,0


## 2. Normalize

CSV trong `Data_raw/` **đã được scale sẵn** (mean ≈ 0, std ≈ 1) → không cần normalize lại khi train.

Khi predict từ UI (giá trị thật như age=58), app sẽ dùng `CONT_STATS` ở trên để transform.

In [4]:
# Kiểm tra: dữ liệu đã scale chưa?
print("Continuous features — mean / std trên train:")
for col in CONT_STATS:
    print(f"  {col:10s}  mean={X_train[col].mean():.3f}  std={X_train[col].std():.3f}")

Continuous features — mean / std trên train:
  age         mean=-0.000  std=1.002
  trestbps    mean=-0.000  std=1.002
  chol        mean=0.000  std=1.002
  thalach     mean=-0.000  std=1.002
  oldpeak     mean=-0.000  std=1.002


## 3. Train Models

In [5]:
def build_models() -> dict:
    models = {
        "DecisionTreeClassifier": DecisionTreeClassifier(max_depth=5, random_state=42),
        "KNeighborsClassifier":   KNeighborsClassifier(n_neighbors=5),
        "GaussianNB":             GaussianNB(),
        "RandomForestClassifier": RandomForestClassifier(
            n_estimators=200, max_depth=6, random_state=42),
        "AdaBoostClassifier": AdaBoostClassifier(
            n_estimators=120, learning_rate=0.8, random_state=42),
        "GradientBoostingClassifier": GradientBoostingClassifier(random_state=42),
    }
    if XGBClassifier:
        models["XGBClassifier"] = XGBClassifier(
            n_estimators=120, max_depth=3, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, eval_metric="logloss", random_state=42,
        )
    # Voting dùng estimator riêng (không share object với model standalone)
    models["VotingClassifier"] = VotingClassifier(
        estimators=[
            ("rf",  RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)),
            ("gb",  GradientBoostingClassifier(random_state=42)),
            ("ada", AdaBoostClassifier(n_estimators=120, learning_rate=0.8, random_state=42)),
        ],
        voting="soft",
    )
    return models


trained = {}
for name, model in build_models().items():
    model.fit(X_train, y_train)
    trained[name] = model
    print(f"✓ {MODEL_NAMES[name]}")

✓ Decision Tree
✓ K-NN
✓ Naive Bayes
✓ Random Forest
✓ AdaBoost
✓ Gradient Boosting
✓ Ensemble (Soft Voting)


## 4. Evaluate trên Validation Set

In [6]:
rows = []
for name, model in trained.items():
    acc = accuracy_score(y_val, model.predict(X_val))
    rows.append({"model": MODEL_NAMES[name], "accuracy": acc})
    print(f"{MODEL_NAMES[name]:30s}  accuracy = {acc:.2%}")

report = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
print(f"\nBest model: {report.iloc[0]['model']} ({report.iloc[0]['accuracy']:.2%})")
report

Decision Tree                   accuracy = 76.67%
K-NN                            accuracy = 93.33%
Naive Bayes                     accuracy = 90.00%
Random Forest                   accuracy = 93.33%
AdaBoost                        accuracy = 93.33%
Gradient Boosting               accuracy = 93.33%
Ensemble (Soft Voting)          accuracy = 93.33%

Best model: K-NN (93.33%)


,model,accuracy
1,K-NN,0.933333
3,Random Forest,0.933333
4,AdaBoost,0.933333
5,Gradient Boosting,0.933333
6,Ensemble (Soft Voting),0.933333
2,Naive Bayes,0.900000
0,Decision Tree,0.766667


In [7]:
# Chi tiết best model
best_key = next(k for k, v in MODEL_NAMES.items() if v == report.iloc[0]["model"])
print(classification_report(y_val, trained[best_key].predict(X_val),
                            target_names=["No HD", "Heart Disease"]))

               precision    recall  f1-score   support

        No HD       1.00      0.88      0.93        16
Heart Disease       0.88      1.00      0.93        14

     accuracy                           0.93        30
    macro avg       0.94      0.94      0.93        30
 weighted avg       0.94      0.93      0.93        30



## 5. Save Models

In [8]:
MODEL_DIR.mkdir(exist_ok=True)

payload = {
    "models":      trained,
    "features":    FEATURES,
    "cont_stats":  CONT_STATS,
    "model_names": MODEL_NAMES,
    "report":      report,
}
joblib.dump(payload, MODEL_DIR / "heart_disease_models.joblib")
joblib.dump(trained[best_key], MODEL_DIR / "best_model.joblib")
report.to_csv(MODEL_DIR / "training_report.csv", index=False)

print(f"✅ Saved → {MODEL_DIR}/")
print("   heart_disease_models.joblib")
print("   best_model.joblib")
print("   training_report.csv")
print("\n→ Giờ chạy: streamlit run app.py")

✅ Saved → /Users/thdziii/Documents/MSA/HW3/models/
   heart_disease_models.joblib
   best_model.joblib
   training_report.csv

→ Giờ chạy: streamlit run app.py
